# Exploratory data analysis for footbal dataset

Data visualisation group project: Group 10

In [1]:
# Import the relevant libraries
import pandas as pd
from sqlalchemy import create_engine
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# Run this file in your data visualisation venv to make sure you have all the right components available

# IMPORTANT!!!
# Make sure you import sqlalchemy into your uv environment before you start, or else this won't work
# Do this in the terminal:
# uv add SQLAlchemy

In [2]:
# Load the data from a db file

# Change the path to the football.db file
PATH_TO_DB = "/Users/qing/Desktop/LBS/Visualization group project/football/football.db"

SQL_ENDPOINT = f"sqlite+pysqlite:///{PATH_TO_DB}" # No need to change this

# Use this format to make a SQL query to the database
# The result will be a Pandas dataframe containing the result of that query
# Use this if you need to join tables together
# Tip: design the query in DBeaver first, because it has a better SQL editor and it understands the relationship between the tables
sql_query = pd.read_sql_query("""
select p.name, ge.description
from players p
join game_events ge on p.player_id = ge.player_id
where p.name = "Lucas Digne" and ge.type = 'Goals'
""",
SQL_ENDPOINT,
dtype_backend="numpy_nullable")

# Use this format to get one of the tables from the database
# The result will be a Pandas dataframe containing the requested table, in this example "players"
# Use this if you just need one of the tables without anything else
# You could also load the csv file in directly if you wanted to, but this is a bit nicer
sql_table = pd.read_sql_table("players", SQL_ENDPOINT, dtype_backend="numpy_nullable")

# Show the contents of the table
sql_table

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,foot,height_in_cm,contract_expiration_date,agent_name,image_url,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,right,184,,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000,30000000
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,left,190,,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000,8000000
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,,,,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000,34500000
3,77,,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,,,,,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000,24500000
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,right,194,,IFM,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000,3000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32596,1375876,,Diego Henrique,Diego Henrique,2024,86209,diego-henrique,Brazil,"Andradina, SP",Brazil,...,left,170,2026-06-30 00:00:00,,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/diego-henrique...,UKR1,FK Livyi Bereh,,
32597,1378362,Orseer,Achihi,Orseer Achihi,2024,1096,orseer-achihi,,,Nigeria,...,,,2029-06-30 00:00:00,Aneke/PMG,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/orseer-achihi/...,BE1,Royal Antwerp Football Club,,
32598,1380311,Prince Amoako,Junior,Prince Amoako Junior,2024,2778,prince-amoako-junior,,,Ghana,...,,,2029-12-31 00:00:00,CAA Stellar,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/prince-amoako-...,DK1,Fodbold Club Nordsjælland,,
32599,1380876,Gabriel Jesus,David,Gabriel Jesus David,2024,1096,gabriel-jesus-david,,,Nigeria,...,,,2025-06-30 00:00:00,Aneke/PMG,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/gabriel-jesus-...,BE1,Royal Antwerp Football Club,,


In [3]:
query = """
SELECT
    strftime('%Y', date) AS year,
    competition_id,
    home_club_formation AS formation
FROM games
WHERE competition_id IN ('ES1','IT1','FR1','GB1','L1')

UNION ALL

SELECT
    strftime('%Y', date) AS year,
    competition_id,
    away_club_formation AS formation
FROM games
WHERE competition_id IN ('ES1','IT1','FR1','GB1','L1');
"""
formation_df = pd.read_sql_query(query, SQL_ENDPOINT, dtype_backend="numpy_nullable")
formation_df


,year,competition_id,formation
0,2012,L1,
1,2012,L1,
2,2012,L1,
3,2012,L1,
4,2012,L1,
...,...,...,...
46313,2025,ES1,4-4-2 double 6
46314,2025,ES1,4-4-2 double 6
46315,2025,ES1,4-3-3 Attacking
46316,2025,ES1,4-4-2


In [4]:
# change all empty string to nan
mask = formation_df['formation'].str.strip() == ''
formation_df.loc[mask, 'formation'] = pd.NA   
formation_df = formation_df.dropna(subset=['formation'])

In [5]:
formation_df.info() # quick check

<class 'pandas.core.frame.DataFrame'>
Index: 42553 entries, 1826 to 46317
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   year            42553 non-null  string
 1   competition_id  42553 non-null  string
 2   formation       42553 non-null  string
dtypes: string(3)
memory usage: 1.3 MB


In [6]:
# count the matches for each year and each competition_id
counts = (
    formation_df
    .groupby(['year', 'competition_id', 'formation'])
    .size()
    .reset_index(name='num_matches') 
)

counts.head()

,year,competition_id,formation,num_matches
0,2013,ES1,3-4-3,1
1,2013,ES1,4-1-4-1,16
2,2013,ES1,4-2-3-1,254
3,2013,ES1,4-3-3,1
4,2013,ES1,4-3-3 Attacking,28


In [7]:
top_5_formation = counts.groupby(['formation'], as_index=False)[['num_matches']].sum().sort_values('num_matches', ascending=False).head(5) # select the top 5 formation
top_5_formation

,formation,num_matches
12,4-2-3-1,12199
17,4-3-3 Attacking,7348
22,4-4-2 double 6,4659
8,3-5-2 flat,3085
3,3-4-2-1,2578


In [8]:
formation_name = top_5_formation['formation'].to_list() # convert to list

total_formation = counts[counts['formation'].isin(formation_name)]

# count the matches for each formation without the specific competition_id
total_formation = (
    total_formation
    .groupby(['year', 'formation'], as_index=False)['num_matches']
    .sum()
)
total_formation

,year,formation,num_matches
0,2013,3-4-2-1,10
1,2013,3-5-2 flat,115
2,2013,4-2-3-1,746
3,2013,4-3-3 Attacking,182
4,2013,4-4-2 double 6,227
...,...,...,...
60,2025,3-4-2-1,166
61,2025,3-5-2 flat,76
62,2025,4-2-3-1,458
63,2025,4-3-3 Attacking,151


In [9]:
# total natches in each season
matches_per_year = (
    formation_df
    .groupby('year')
    .size()
    .reset_index(name='total_matches')
)
matches_per_year

,year,total_matches
0,2013,1742
1,2014,3612
2,2015,3692
3,2016,3624
4,2017,3754
5,2018,3520
6,2019,3646
7,2020,3138
8,2021,3975
9,2022,3374


In [10]:
total_formation = total_formation.merge(matches_per_year, on='year', how='left')
total_formation['usage_rate'] = (
    total_formation['num_matches'] / total_formation['total_matches'] * 100
)

total_formation

,year,formation,num_matches,total_matches,usage_rate
0,2013,3-4-2-1,10,1742,0.574053
1,2013,3-5-2 flat,115,1742,6.601607
2,2013,4-2-3-1,746,1742,42.824340
3,2013,4-3-3 Attacking,182,1742,10.447761
4,2013,4-4-2 double 6,227,1742,13.030999
...,...,...,...,...,...
60,2025,3-4-2-1,166,1206,13.764511
61,2025,3-5-2 flat,76,1206,6.301824
62,2025,4-2-3-1,458,1206,37.976783
63,2025,4-3-3 Attacking,151,1206,12.520730


In [11]:
# plot the graph of formation over time
fig = px.line(
    total_formation,
    x = "year",
    y = "usage_rate",
    color = "formation",
    markers = True,
    labels = {
        "year": "year",
        "usage_rate": "usage rate (%)",
        "formation": "Formation"
    }
)

fig.update_traces(hovertemplate="year=%{x}<br>usage=%{y:.1f}%")

# descriptive title
fig.update_layout(
    title=dict(
        text="Top 5 formations over time in the top 5 leagues",
        x=0.05,          
        xanchor="left"
    ),
    margin=dict(t=90)  
)

# subtitle
fig.add_annotation(
    x=0.5, y=1.06,          
    xref="paper", yref="paper",
    text="The 4-2-3-1 formation is the most popular",
    showarrow=False,
    xanchor="center",
    font=dict(size=12)
)

fig.show()

The chart shows that 4-2-3-1 remains the most popular formation over time, starting with a very high usage rate in 2013, then declining each season until it reaches a low point around 2019, before recovering strongly in the following years. 4-3-3 Attacking acts as the main alternative with a solid but more volatile share, while 4-4-2 double 6 gradually falls from a stable mid teen percentage to only a few percent by the end. In contrast, three at the back systems gain ground, with 3-4-2-1 growing from almost zero to a level comparable to 4-3-3 Attacking. Overall the pattern suggests a shift away from traditional 4-4-2 double 6 toward flexible 4-2-3-1 and modern three centre back shapes, with 2019 standing out as the key turning point when the dominance of 4-2-3-1 reaches its minimum before the later resurgence.